# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema, accessible via a provided JSON-LD URL. All dataset entities (record sets, fields, columns) are referenced by their `@id` per FAIR best practices.


In [ ]:
# Ensure the required library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Let's load the metadata and records from the Croissant dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset title:", metadata.name)
print("Description:", metadata.description)


## 2. Data Overview
We'll enumerate available record sets, fields, and their `@id`s for reference.

In [ ]:
# Display an overview of the dataset's record sets
print("Record Sets in the dataset:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set['@id']}")
    print(f"  name: {record_set['name']}")
    print(f"  description: {record_set.get('description', '')}")
    print(f"  Fields/Columns:")
    for field in record_set.get('fields', []):
        print(f"    - @id: {field['@id']} | name: {field['name']} | dataType: {field.get('dataType','n/a')}")
    print()
# Collect all record set @ids for later use
record_set_ids = [r['@id'] for r in metadata.record_sets]
# For demonstration, pick the first record set as default
if len(record_set_ids) == 0:
    raise ValueError('No record sets found in this dataset!')
main_record_set_id = record_set_ids[0]


## 3. Data Extraction
Let's load records from all the available record sets into pandas DataFrames, referencing each by its `@id`.

We'll preview the main record set.

In [ ]:
# Load all records for each record set into a DataFrame
dataframes = {}

print('Loading records from each record set...')
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for {rec_id}")
# Inspect columns and first rows for the main record set
print(f"\nColumns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic analysis using the DataFrame from the main record set.

We'll demonstrate filtering, normalization, and grouping using numeric and categorical fields, with columns always referenced by their `@id`.

In [ ]:
# -- Customization required here depending on actual field names --
# We'll try to automatically detect a numeric and group field for demonstration.
df = dataframes[main_record_set_id]

print('Columns and sample data:')
df_head = df.head()
display(df_head)

# Identify a valid numeric field (float or int)
numeric_field_id = None
for col in df.columns:
    # Try to infer if the column is numeric
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    # Try to parse numerics if data imported as strings
    for col in df.columns:
        try:
            sample = pd.to_numeric(df[col], errors='coerce')
            if sample.notnull().sum() > 0:
                df[col] = sample
                numeric_field_id = col
                break
        except Exception:
            continue
print(f"Numeric Field Chosen (by @id): {numeric_field_id}")
# Filter by a threshold
threshold = 10

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records in '{main_record_set_id}' where '{numeric_field_id}' > {threshold}:")
display(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"First (normalized) values for filtered '{numeric_field_id}':")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Look for a likely categorical or group-by field (different from the numeric)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() > 1 and df[col].nunique() < len(df) // 2:
        group_field_id = col
        break
if group_field_id:
    print(f"\nGrouping by field: {group_field_id}\n")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships. We'll show a histogram of the selected numeric field and, if group field exists, a bar plot of means per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping field is available, show mean per group
if group_field_id:
    plt.figure(figsize=(10,6))
    order = grouped_df.sort_values(numeric_field_id, ascending=False)[group_field_id]
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, order=order)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## 6. Conclusion
In this notebook we:
- Loaded the FAIR² mass spectrometry dataset using the `mlcroissant` library;
- Inspected dataset metadata, record sets, and fields via their `@id`s;
- Loaded record data into DataFrames and explored fields;
- Demonstrated basic filtering, normalization, grouping, and visualization for quick EDA.

**Next Steps:**
- Refine EDA with domain-specific questions (e.g. which proteins are most abundant by sample/condition?);
- Use field `@id`s as stable references for all programmatic data operations;
- Integrate these data with external resources or for downstream statistical/machine learning workflows.